# Problem 040:  Champernowne's Constant

> An irrational decimal fraction is created by concatenating the positive integers:
> $$0.12345678910{\color{red}\mathbf 1}112131415161718192021\cdots$$
>
> It can be seen that the $12$th digit of the fractional part is $1$.
>
> If $d_n$ represents the $n$th digit of the fractional part, find the value of the following expression.
> $$d_1 \times d_{10} \times d_{100} \times d_{1000} \times d_{10000} \times d_{100000} \times d_{1000000}$$

## Naive Solution $\mathcal O\left(N\right)$

This is more of a "how lazy can you get".  Why not just construct the decimal representation using all the numbers up to the largest index in the list?  It's overkill, to say the least, but easy enough.

In [61]:
%%time

from math import prod

def champernowne(n: int) -> str:
    return "".join(map(str, range(1, n)))

def p040_naive(N: list[int]) -> int:
    c_str = champernowne(max(N))
    return prod(int(c_str[n-1]) for n in N)
    
N = [pow(10,i) for i in range(7)]
ans = p040_naive(N)

print(ans)

210
CPU times: user 324 ms, sys: 35 ms, total: 359 ms
Wall time: 365 ms


## Faster Solution $\mathcal O\left(\ln N \right)$

Let's do the problem "correctly".  First, find the pattern in the number of digits in the decimal expansion using some power of $10$  number of values.  There are $9$ single digit numbers, $90$ double digit numbers, $900$ triple digit numbers, and $9 \times 10^{m-1}$ $m$-digit numbers.  Now, step through the number $m$ starting at $m=1$ and count how many digits are used by adding $m \times 9 \times 10^{m-1}$ until you reach a count larger than the target index $n$.  The resulting $m$ tells you the number of digits in the number of which your index will be a part.  Namely, the number the index is involved in is $10^{m-1} + \lfloor (n - c - 1) / m \rfloor$ and the index will be $(n-c-1)\% m$, where $c$ is the remaining number of digits after removing all those from numbers less than $10^{m-1}$.

This algorithm has a time scaling of $\mathcal O (\ln N)$ because of the iterative approach of finding the number of digits in the desired number.

In [64]:
%%time

from math import prod

def champernowne(n: int) -> int:
    cnt = 9
    pow10 = 1
    m = 1
    while cnt < n:
        m += 1
        pow10 *= 10
        cnt += m * 9 * pow10
        
    cnt -= m * 9 * pow10
    return int(str(pow10 + (n - cnt - 1) // m)[(n - cnt - 1) % m])

def p040(N: list[int]) -> int:
    return prod(champernowne(n) for n in N)
    
N = [pow(10,i) for i in range(7)]
ans = p040(N)

print(ans)

210
CPU times: user 212 μs, sys: 13 μs, total: 225 μs
Wall time: 210 μs


It is interesting to consider if there is a more direct way of finding the number of which the index is a part.  Namely, consider the sum generated in the iterative approach above:
$$c_m = \sum_{k = 1}^{m} 9k10^{k-1} = \sum_{k = 1}^{m} 10^{m} - 10^{k-1} = m10^m - \frac{10^m-1}{9}$$,
where the second sum here can be thought of as the number of digits in the $1$'s place, plus $10$'s place, and so on.  The last relation is the result of the geometric sequence.

The goal would be to invert this equation to find the number $m$ directly for a given $N$.  Unfortunately, this is not trivial and would require the use of the Lambert W equation.  That said, there would be ways of putting in an initial guess of $m$ and refining the value in a way that would be faster than $\ln N$ time, but seems like too much guilding of the lily.

Also, what makes it even less attractive, is that the algorithm would still have a time scaling of $\ln N$ because once you find the number of which the index is a part, you still need to find an arbitrary digit in that number.  This step will still have $\ln N$ timing, so the asymptotics would be unchanged after all the effort to find $m$ more directly.